# Anarkali 150M live test

This notebook runs the recovered **Ettin 150M distilled** checkpoint and prints its real decision probabilities. Select a GPU runtime before running the cells.

In [ ]:
# Cell 1: Clone the exact branch and install Anarkali
!git clone --depth 1 --branch claude/distill-brio-data https://github.com/ToufiqQureshi/anarkali.git /content/anarkali
%cd /content/anarkali
!pip -q install -e .


In [ ]:
# Cell 2: Confirm that Colab assigned a GPU
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then reconnect.'
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Cell 3: Upload ONLY the 150M best.pt file (about 569 MB).
# Local path: .kaggle-run/eval-v3/anarkali/artifacts/distill-run/distilled-150m/best.pt
from google.colab import files
from pathlib import Path
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Select best.pt in the upload picker.')
uploaded_name = next(iter(uploaded))
if not uploaded_name.endswith('.pt'):
    raise ValueError('Upload best.pt only, not results.zip.')
checkpoint = Path('/content') / uploaded_name
print(f'Checkpoint ready: {checkpoint} ({checkpoint.stat().st_size / 1e6:.1f} MB)')


In [ ]:
# Cell 4: Run Anarkali's built-in visible release-decision example.
!python scripts/demo_recovered_checkpoint.py --checkpoint "{checkpoint}" --device cuda


In [ ]:
# Cell 5: Test your own decision. Edit the state, question, and option text, then run.
import json
state = {
    'task': 'Release Anarkali safely.',
    'constraints': ['Do not expose credentials.', 'Run tests before publishing.'],
    'recent_actions': ['Recovered the 150M checkpoint from Kaggle.'],
}
question = {
    'type': 'choice',
    'instructions': 'What should happen next?',
    'criteria': {
        'A': 'Run the held-out evaluation and inspect its results.',
        'B': 'Publish without evaluating the recovered model.',
        'C': 'Delete saved checkpoints and train from scratch.',
    },
}
Path('/content/state.json').write_text(json.dumps(state), encoding='utf-8')
Path('/content/question.json').write_text(json.dumps(question), encoding='utf-8')
!python scripts/demo_recovered_checkpoint.py --checkpoint "{checkpoint}" --device cuda --state-json /content/state.json --question-json /content/question.json
